In [ ]:
import os
import json
from pathlib import Path
from typing import Optional, Callable


import torch
import librosa
from torch.utils.data import Dataset


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ROOT_DIR = Path.home() / "workspace"
DATA_DIR = ROOT_DIR / "dataset" / "Clotho-Moment"

In [ ]:
AUDIO_DIR = DATA_DIR / "32ch_audio_dev" / "eigen_dev"
LABEL_DIR = DATA_DIR / "labels_dev"

In [ ]:
print(f"Within audio directory: {os.listdir(AUDIO_DIR)}")
print(f"Within label directory: {os.listdir(LABEL_DIR)}")

In [ ]:
AUDIO_DIR = AUDIO_DIR / "dev-train-tau"
LABEL_DIR = LABEL_DIR / "dev-train-tau"

In [ ]:
sorted(os.listdir(AUDIO_DIR))

In [ ]:
sorted(os.listdir(LABEL_DIR))

In [ ]:
label_files = os.listdir(LABEL_DIR)
audio_files = os.listdir(AUDIO_DIR)

# Create a dictionary for quick lookup of labels by base name
label_dict = {}
for label in label_files:
    base_name = label.replace("_std.json", "")  # Remove suffix and extension
    label_dict[base_name] = label

# Pair audio files with matching labels
paired_files = []
for audio in audio_files:
    base_name = audio.replace(".wav", "")  # Remove extension
    if base_name in label_dict:
        paired_files.append((audio, label_dict[base_name]))

# Optional: Print the pairs for verification
for audio, label in paired_files:
    print(f"Audio: {audio} -> Label: {label}")

## Torch Dataset Implementation

In [ ]:
class ClothoMomentDataset(Dataset):
    def __init__(
        self,
        audio_dir: Path,
        label_dir: Path,
        sample_rate: int = 44100,
        *,
        transform: Optional[Callable] = None,
    ):
        self.audio_dir = audio_dir
        self.label_dir = label_dir
        self.sample_rate = sample_rate
        self.transform = transform

        self.audio_files = sorted(list(self.audio_dir.glob("*.wav")))
        self.label_files = sorted(list(self.label_dir.glob("*.json")))

        self.pair_files = [
            (audio, label)
            for audio in self.audio_files
            for label in self.label_files
            if audio.stem == label.stem.replace("_std", "")
        ]

    def __len__(self):
        return len(self.pair_files)

    def __getitem__(self, idx):
        audio_path, label_path = self.pair_files[idx]

        # Load audio
        waveform, sample_rate = librosa.load(audio_path, sr=self.sample_rate)

        # Load labels
        with open(label_path, "r") as f:
            labels = json.load(f)

        if self.transform:
            waveform = self.transform(waveform)

        return {
            "waveform": waveform,
            "sample_rate": sample_rate,
            "labels": labels,
        }

In [ ]:
dataset = ClothoMomentDataset(AUDIO_DIR, LABEL_DIR)
print(f"Dataset size: {len(dataset)}")

In [ ]:
sample = dataset[0]
print(f"Sample keys: {sample.keys()}")
print(f"Waveform shape: {sample['waveform'].shape}")
print(f"Sample rate: {sample['sample_rate']}")
print(f"Labels: {sample['labels']}")